In [23]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Literal

In [ ]:
df_info = {
    'Folio': 'Conteo del 1 al 10000',
    'Cosecha': 'Cada folio tiene la posibilidad de comenzar en jun-22, y extenderse hasta oct-25',
    'Capital': 'Entero desde 120k hasta 100M',
    'BGI2+': 'Entero desde 0 hasta 100M',
    'BGI3+': 'Entero desde 0 hasta 100M',
    'BGI4+': 'Entero desde 0 hasta 100M',
    'BGI5+': 'Entero desde 0 hasta 100M',
    'CAST': 'Entero desde 120k hasta 100M',
    'BGI2+_CONTEO_EVER': 'Si hay monto en BGI2+ o en BGI mayor',
    'BGI3+_CONTEO_EVER': 'Si hay monto en BGI3+ o en BGI mayor',
    'BGI4+_CONTEO_EVER': 'Si hay monto en BGI4+ o en BGI mayor',
    'BGI5+_CONTEO_EVER': 'Si hay monto en BGI5+ o en BGI mayor',
    'CAST_CONTEO': 'Si hay monto en CAST', 
    'Perfil': ['A', 'B', 'C', 'D', 'E', 'F'],
    'Ciudad Atria': ['Cancún', 'Cdmx', 'Celaja', 
                     'Chihuahua', 'Ciudad Juarez', 'Guadlajara', 
                     'Irapuato', 'León', 'Merida',
                     'Monterrey', 'Pachuca', 'Queretaro',
                     'Torreón'],
    'Plazo': ['12', '24', '36', '48', '60'],
    'Antiguedad Unidad': ['<4', '4 a <6', '6 a <8', '>=8'],
    'Laboratorios': ['W01', 'W2', 'W3', 'Z01', 'Z02', 'Z03', 'Z04', 'Normal'],
    'Notificaciones': ['PE A', 'PE B', 'GG 1', 'GG 2', 'GG 3', 'Normal'],
    'Atria Plus': ['PE', 'GG', 'Score', 'Normal'],
    'MMX 6': ['MOP1',  'MOP2',  'MOP3',  'MOP4',  'MOP5+'],
    'MMX 12': ['MOP1',  'MOP2',  'MOP3',  'MOP4',  'MOP5+'],
    'MMX 24': ['MOP1',  'MOP2',  'MOP3',  'MOP4',  'MOP5+'],
    'Score generico de bancos': 'Entero del 200 al 900',
    'Cluster Mora': ['BGI2+', 'BGI3+', 'BGI4+', 'BGI5+', 'CAST'],
    'Cluster Experiencia': ['Exp1', 'Exp2', 'Exp3', 'Exp4', 'Exp5'],
    'Score Buro': ['>0 a <556', '556 a <613', '613 a <663', '>663'],
    '% Utilizacion Revolvente': ['NULL', '>=0 a <0.068', 
                                 '>=0.068 a <0.305', '>=0.305 a <0.621', 
                                 '>=0.621663'],
    'Endeudamiento': ['NULL', '<0.019', '>=0.019 a <0.067', 
                      '>=0.067 a <0.171', '>=0.171 a <0.305', 
                      '>=0.305'],
}

cols_2 = {f'MOB{x}': 'Porcentaje random del 0% al 20%' for x in range(1, 42)}

df_info.update(cols_2)

# print(df_info.keys())
df = pd.DataFrame(columns=df_info.keys())
df

,Folio,Cosecha,Capital,BGI2+,BGI3+,BGI4+,BGI5+,CAST_VAL,BGI2+_CONTEO_EVER,BGI3+_CONTEO_EVER,...,MOB32,MOB33,MOB34,MOB35,MOB36,MOB37,MOB38,MOB39,MOB40,MOB41


# Para df en formato Wide

In [ ]:
def build_dummy_df(n: int = 10_000, seed: int = 42, mora_type: str = "BGI4+") -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    ALLOWED_MORA = {"BGI2+", "BGI3+", "BGI4+", "BGI5+", "CAST"}
    if mora_type not in ALLOWED_MORA:
        raise ValueError(f"mora_type debe ser uno de {sorted(ALLOWED_MORA)}. Recibido: {mora_type}")

    # --- Catálogos ---
    perfiles = ['A', 'B', 'C', 'D', 'E', 'F']
    ciudades = ['Cancún', 'CDMX', 'Celaja', 'Chihuahua', 'Ciudad Juarez', 'Guadlajara',
                'Irapuato', 'León', 'Merida', 'Monterrey', 'Pachuca', 'Queretaro', 'Torreón']
    plazos = ['12', '24', '36', '48', '60']
    antiguedad = ['<4', '4 a <6', '6 a <8', '>=8']
    labs = ['W01', 'W2', 'W3', 'Z01', 'Z02', 'Z03', 'Z04', 'Normal']
    notif = ['PE A', 'PE B', 'GG 1', 'GG 2', 'GG 3', 'Normal']
    atria_plus = ['PE', 'GG', 'Score', 'Normal']
    mop = ['MOP1', 'MOP2', 'MOP3', 'MOP4', 'MOP5+']
    cluster_exp = ['Exp1', 'Exp2', 'Exp3', 'Exp4', 'Exp5']
    score_buro = ['>0 a <556', '556 a <613', '613 a <663', '>663']
    util_rev = ['NULL', '>=0 a <0.068', '>=0.068 a <0.305', '>=0.305 a <0.621', '>=0.621663']
    endeud = ['NULL', '<0.019', '>=0.019 a <0.067', '>=0.067 a <0.171', '>=0.171 a <0.305', '>=0.305']

    # --- Cosecha (mes inicio) ---
    # Meses entre 2022-06 y 2025-10
    months = pd.period_range("2022-06", "2025-10", freq="M").astype(str)
    cosecha = rng.choice(months, size=n, replace=True)

    # --- Folio ---
    folio = np.arange(1, n + 1, dtype=int)

    # --- Capital ---
    capital = rng.integers(120_000, 1_200_000, size=n, dtype=np.int64)

    # --- Generar monto de mora elegido (siempre <= Capital) ---
    mora_amount = (rng.random(n) * capital).astype(np.int64)

    # Nombre de la columna mora (ej. "BGI4+" o "CAST")
    mora_col = mora_type

    # Nombre de la columna EVER (según mora)
    if mora_type == "CAST":
        ever_col = "CAST_CONTEO"
    else:
        ever_col = f"{mora_type}_CONTEO_EVER"

    # EVER: bandera si hay monto en la mora elegida
    ever = (mora_amount > 0).astype(np.int8)

    # --- MOB1..MOB41 (porcentaje 0..20) ---
    mob = rng.random((n, 41)) * 20.0
    mob_cols = {f"MOB{i}": mob[:, i - 1] for i in range(1, 42)}

    # --- DataFrame base ---
    df = pd.DataFrame({
        "Folio": folio,
        "Cosecha": cosecha,
        "Capital": capital,
        mora_col: mora_amount,
        ever_col: ever,
        "Perfil": rng.choice(perfiles, size=n),
        "Ciudad Atria": rng.choice(ciudades, size=n),
        "Plazo": rng.choice(plazos, size=n),
        "Antiguedad Unidad": rng.choice(antiguedad, size=n),
        "Laboratorios": rng.choice(labs, size=n),
        "Notificaciones": rng.choice(notif, size=n),
        "Atria Plus": rng.choice(atria_plus, size=n),
        "MMX 6": rng.choice(mop, size=n),
        "MMX 12": rng.choice(mop, size=n),
        "MMX 24": rng.choice(mop, size=n),
        "Score generico de bancos": rng.integers(200, 901, size=n, dtype=np.int64),

        # Si solo hay una mora en el df dummy, esto puede fijarse así:
        "Cluster Mora": np.full(n, "CAST" if mora_col == "CAST" else mora_col, dtype=object),

        "Cluster Experiencia": rng.choice(cluster_exp, size=n),
        "Score Buro": rng.choice(score_buro, size=n),
        "% Utilizacion Revolvente": rng.choice(util_rev, size=n),
        "Endeudamiento": rng.choice(endeud, size=n),
    })

    # Agregar MOBs
    for col, vals in mob_cols.items():
        df[col] = vals

    return df


In [26]:
df_dummy_bgi4_wide = build_dummy_df(n=5_000, seed=123, mora_type="BGI4+")
df_dummy_bgi4_wide.head()

,Folio,Cosecha,Capital,BGI4+,BGI4+_CONTEO_EVER,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,...,MOB32,MOB33,MOB34,MOB35,MOB36,MOB37,MOB38,MOB39,MOB40,MOB41
0,1,2022-06,151330,89756,1,C,León,24,6 a <8,W2,...,3.004006,4.288380,0.299734,11.482095,11.249582,12.621936,15.314758,17.270715,11.247619,19.985977
1,2,2024-09,776191,274360,1,C,Irapuato,36,<4,Z04,...,16.048905,5.657550,8.789486,17.112009,14.032672,11.205997,5.390026,19.468749,1.219984,2.544476
2,3,2024-06,576905,193999,1,F,Merida,48,4 a <6,Z04,...,0.660985,17.689881,13.393486,2.883985,7.389792,8.243152,12.364393,0.133662,3.908697,17.756083
3,4,2022-08,271885,108681,1,D,Torreón,48,<4,W2,...,10.699411,3.379737,2.158769,19.774465,16.312961,0.813862,12.953771,16.601891,5.427033,15.187290
4,5,2025-07,247209,226309,1,A,Guadlajara,36,<4,W2,...,15.591901,15.520974,19.004863,0.603616,6.096702,4.656255,18.707334,8.732129,13.878100,1.486468


In [27]:
df_dummy_bgi3_wide = build_dummy_df(n=5_000, seed=123, mora_type="BGI3+")
df_dummy_bgi3_wide.head()

,Folio,Cosecha,Capital,BGI3+,BGI3+_CONTEO_EVER,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,...,MOB32,MOB33,MOB34,MOB35,MOB36,MOB37,MOB38,MOB39,MOB40,MOB41
0,1,2022-06,151330,89756,1,C,León,24,6 a <8,W2,...,3.004006,4.288380,0.299734,11.482095,11.249582,12.621936,15.314758,17.270715,11.247619,19.985977
1,2,2024-09,776191,274360,1,C,Irapuato,36,<4,Z04,...,16.048905,5.657550,8.789486,17.112009,14.032672,11.205997,5.390026,19.468749,1.219984,2.544476
2,3,2024-06,576905,193999,1,F,Merida,48,4 a <6,Z04,...,0.660985,17.689881,13.393486,2.883985,7.389792,8.243152,12.364393,0.133662,3.908697,17.756083
3,4,2022-08,271885,108681,1,D,Torreón,48,<4,W2,...,10.699411,3.379737,2.158769,19.774465,16.312961,0.813862,12.953771,16.601891,5.427033,15.187290
4,5,2025-07,247209,226309,1,A,Guadlajara,36,<4,W2,...,15.591901,15.520974,19.004863,0.603616,6.096702,4.656255,18.707334,8.732129,13.878100,1.486468


In [ ]:
df_dummy_CAST_wide = build_dummy_df(n=5_000, seed=123, mora_type="CAST")
df_dummy_CAST_wide.head()

,Folio,Cosecha,Capital,CAST_VAL,CAST_CONTEO,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,...,MOB32,MOB33,MOB34,MOB35,MOB36,MOB37,MOB38,MOB39,MOB40,MOB41
0,1,2022-06,151330,89756,1,C,León,24,6 a <8,W2,...,3.004006,4.288380,0.299734,11.482095,11.249582,12.621936,15.314758,17.270715,11.247619,19.985977
1,2,2024-09,776191,274360,1,C,Irapuato,36,<4,Z04,...,16.048905,5.657550,8.789486,17.112009,14.032672,11.205997,5.390026,19.468749,1.219984,2.544476
2,3,2024-06,576905,193999,1,F,Merida,48,4 a <6,Z04,...,0.660985,17.689881,13.393486,2.883985,7.389792,8.243152,12.364393,0.133662,3.908697,17.756083
3,4,2022-08,271885,108681,1,D,Torreón,48,<4,W2,...,10.699411,3.379737,2.158769,19.774465,16.312961,0.813862,12.953771,16.601891,5.427033,15.187290
4,5,2025-07,247209,226309,1,A,Guadlajara,36,<4,W2,...,15.591901,15.520974,19.004863,0.603616,6.096702,4.656255,18.707334,8.732129,13.878100,1.486468


In [29]:
df_dummy_bgi3_wide.to_csv("dummy_output_data/df_dummy_bgi3_wide.csv", index=False)
df_dummy_bgi4_wide.to_csv("dummy_output_data/df_dummy_bgi4_wide.csv", index=False)
df_dummy_CAST_wide.to_csv("dummy_output_data/df_dummy_CAST_wide.csv", index=False)
print('Data Frames en formato wide guardados')

Data Frames en formato wide guardados


# Para df en formato Long

In [ ]:
def build_dummy_df_long(
    n: int = 10_000,
    seed: int = 42,
    mora_type: str = "BGI4+",
    mob_max: int = 41,
) -> pd.DataFrame:
    """
    Genera dataset dummy en formato LONG: 1 fila = (Folio, MOB).
    Incluye columnas de atributos (Perfil, Ciudad, etc.) + una sola columna de mora + una sola columna EVER
    + columnas: MOB (int) y pct (float 0..20).
    """
    rng = np.random.default_rng(seed)

    ALLOWED_MORA = {"BGI2+", "BGI3+", "BGI4+", "BGI5+", "CAST"}
    if mora_type not in ALLOWED_MORA:
        raise ValueError(f"mora_type debe ser uno de {sorted(ALLOWED_MORA)}. Recibido: {mora_type}")

    # --- Catálogos ---
    perfiles = ['A', 'B', 'C', 'D', 'E', 'F']
    ciudades = ['Cancún', 'Cdmx', 'Celaja', 'Chihuahua', 'Ciudad Juarez', 'Guadlajara',
                'Irapuato', 'León', 'Merida', 'Monterrey', 'Pachuca', 'Queretaro', 'Torreón']
    plazos = ['12', '24', '36', '48', '60']
    antiguedad = ['<4', '4 a <6', '6 a <8', '>=8']
    labs = ['W01', 'W2', 'W3', 'Z01', 'Z02', 'Z03', 'Z04', 'Normal']
    notif = ['PE A', 'PE B', 'GG 1', 'GG 2', 'GG 3', 'Normal']
    atria_plus = ['PE', 'GG', 'Score', 'Normal']
    mop = ['MOP1', 'MOP2', 'MOP3', 'MOP4', 'MOP5+']
    cluster_exp = ['Exp1', 'Exp2', 'Exp3', 'Exp4', 'Exp5']
    score_buro = ['>0 a <556', '556 a <613', '613 a <663', '>663']
    util_rev = ['NULL', '>=0 a <0.068', '>=0.068 a <0.305', '>=0.305 a <0.621', '>=0.621663']
    endeud = ['NULL', '<0.019', '>=0.019 a <0.067', '>=0.067 a <0.171', '>=0.171 a <0.305', '>=0.305']

    # --- Cosecha (YYYY-MM) ---
    # months = pd.period_range("2022-06", "2025-10", freq="M").astype(str)
    months = pd.period_range("2022-06", "2025-10", freq="M")
    cosecha = rng.choice(months, size=n, replace=True)
    cosecha_date = pd.PeriodIndex(cosecha, freq="M").to_timestamp()

    # --- Folio ---
    folio = np.arange(1, n + 1, dtype=np.int64)

    # --- Capital ---
    capital = rng.integers(120_000, 1_200_000, size=n, dtype=np.int64)

    # --- Mora + EVER (una sola) ---
    mora_col = mora_type
    mora_amount = (rng.random(n) * capital).astype(np.int64)

    if mora_type == "CAST":
        ever_col = "CAST_CONTEO"
        cluster_mora_val = "CAST"
    else:
        ever_col = f"{mora_type}_CONTEO_EVER"
        cluster_mora_val = mora_type

    ever = (mora_amount > 0).astype(np.int8)

    # --- Tabla base (folio-level) ---
    df_base = pd.DataFrame({
        "Folio": folio,
        "Cosecha": cosecha_date,
        "Capital": capital,
        mora_col: mora_amount,
        ever_col: ever,
        "Perfil": rng.choice(perfiles, size=n),
        "Ciudad Atria": rng.choice(ciudades, size=n),
        "Plazo": rng.choice(plazos, size=n),
        "Antiguedad Unidad": rng.choice(antiguedad, size=n),
        "Laboratorios": rng.choice(labs, size=n),
        "Notificaciones": rng.choice(notif, size=n),
        "Atria Plus": rng.choice(atria_plus, size=n),
        "MMX 6": rng.choice(mop, size=n),
        "MMX 12": rng.choice(mop, size=n),
        "MMX 24": rng.choice(mop, size=n),
        "Score generico de bancos": rng.integers(200, 901, size=n, dtype=np.int64),
        "Cluster Mora": np.full(n, cluster_mora_val, dtype=object),
        "Cluster Experiencia": rng.choice(cluster_exp, size=n),
        "Score Buro": rng.choice(score_buro, size=n),
        "% Utilizacion Revolvente": rng.choice(util_rev, size=n),
        "Endeudamiento": rng.choice(endeud, size=n),
    })

    # --- LONG: replicar filas por MOB ---
    mobs = np.arange(1, mob_max + 1, dtype=np.int16)
    df_long = df_base.loc[df_base.index.repeat(mob_max)].reset_index(drop=True)
    df_long["MOB"] = np.tile(mobs, n)

    # pct (0..20)
    df_long["pct"] = (rng.random(n * mob_max) * 20.0).astype(float)

    return df_long

In [31]:
df_dummy_bgi4_long = build_dummy_df_long(n=4_500, seed=123, mora_type="BGI4+", mob_max=41)
df_dummy_bgi4_long.head(20)

,Folio,Cosecha,Capital,BGI4+,BGI4+_CONTEO_EVER,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,...,MMX 12,MMX 24,Score generico de bancos,Cluster Mora,Cluster Experiencia,Score Buro,% Utilizacion Revolvente,Endeudamiento,MOB,pct
0,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,1,17.331383
1,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,2,0.541315
2,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,3,11.934600
3,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,4,10.209965
4,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,5,16.688253
5,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,6,4.668121
6,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,7,9.283556
7,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,8,11.858692
8,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,9,18.708449
9,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI4+,Exp4,>663,>=0.621663,NULL,10,8.306143


In [32]:
df_dummy_bgi3_long = build_dummy_df_long(n=4_500, seed=123, mora_type="BGI3+", mob_max=41)
df_dummy_bgi3_long.head(20)

,Folio,Cosecha,Capital,BGI3+,BGI3+_CONTEO_EVER,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,...,MMX 12,MMX 24,Score generico de bancos,Cluster Mora,Cluster Experiencia,Score Buro,% Utilizacion Revolvente,Endeudamiento,MOB,pct
0,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,1,17.331383
1,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,2,0.541315
2,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,3,11.934600
3,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,4,10.209965
4,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,5,16.688253
5,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,6,4.668121
6,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,7,9.283556
7,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,8,11.858692
8,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,9,18.708449
9,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,BGI3+,Exp4,>663,>=0.621663,NULL,10,8.306143


In [ ]:
df_dummy_CAST_long = build_dummy_df_long(n=4_500, seed=123, mora_type="CAST", mob_max=41)
df_dummy_CAST_long.head(20)

,Folio,Cosecha,Capital,CAST_VAL,CAST_CONTEO,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,...,MMX 12,MMX 24,Score generico de bancos,Cluster Mora,Cluster Experiencia,Score Buro,% Utilizacion Revolvente,Endeudamiento,MOB,pct
0,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,1,17.331383
1,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,2,0.541315
2,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,3,11.934600
3,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,4,10.209965
4,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,5,16.688253
5,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,6,4.668121
6,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,7,9.283556
7,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,8,11.858692
8,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,9,18.708449
9,1,2022-06-01,1112783,264094,1,E,Chihuahua,12,>=8,W2,...,MOP5+,MOP4,773,CAST,Exp4,>663,>=0.621663,NULL,10,8.306143


In [34]:
df_dummy_bgi3_long.to_csv("dummy_output_data/df_dummy_bgi3_long.csv", index=False)
df_dummy_bgi4_long.to_csv("dummy_output_data/df_dummy_bgi4_long.csv", index=False)
df_dummy_CAST_long.to_csv("dummy_output_data/df_dummy_CAST_long.csv", index=False)
print('Data Frames en formato long guardados')

Data Frames en formato long guardados


In [35]:
# def make_demo_long_all(df_bgi3: pd.DataFrame, df_bgi4: pd.DataFrame, df_cast: pd.DataFrame) -> pd.DataFrame:
#     # 1) Etiquetar cada dataset
#     a = df_bgi3.copy()
#     a["tipo_mora"] = "BGI3+"

#     b = df_bgi4.copy()
#     b["tipo_mora"] = "BGI4+"

#     c = df_cast.copy()
#     c["tipo_mora"] = "CAST"  # para CAST_VAL

#     # 2) Concatenar
#     df_all = pd.concat([a, b, c], ignore_index=True)

#     # 3) Llave única por fila (folio + mora + mob)
#     df_all["folio_mora_id"] = (
#         df_all["Folio"].astype(str) + "|" +
#         df_all["tipo_mora"].astype(str)
#     )

#     # 4) Sanity: tipos recomendados
#     df_all["MOB"] = df_all["MOB"].astype(int)
#     df_all["pct"] = df_all["pct"].astype(float)

#     return df_all

In [ ]:
def _normalize_one_long(df: pd.DataFrame, *, tipo_mora: str) -> pd.DataFrame:
    df = df.copy()
    df["tipo_mora"] = tipo_mora

    # Detectar columnas de mora y ever
    if tipo_mora == "CAST":
        mora_col = "CAST"
        ever_col = "CAST_CONTEO"
    else:
        mora_col = tipo_mora          # "BGI3+" o "BGI4+"
        ever_col = f"{tipo_mora}_CONTEO_EVER"

    required = ["Folio", "MOB", "pct", mora_col, ever_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"En df ({tipo_mora}) faltan columnas {missing}. "
            f"Disponibles: {list(df.columns)}"
        )

    # Normalizar a esquema común
    df["mora_amount"] = df[mora_col].astype(float)
    df["mora_ever"] = df[ever_col].astype("Int64")

    # Limpiar columnas específicas de mora (para BI)
    drop_candidates = [
        "BGI3+", "BGI3+_CONTEO_EVER",
        "BGI4+", "BGI4+_CONTEO_EVER",
        "CAST", "CAST_CONTEO",
    ]
    drop_cols = [c for c in drop_candidates if c in df.columns]
    df = df.drop(columns=drop_cols)

    # Llaves auxiliares
    df["folio_mora_id"] = df["Folio"].astype(str) + "|" + df["tipo_mora"]
    df["folio_mora_mob_id"] = (
        df["folio_mora_id"] + "|" + df["MOB"].astype(int).astype(str)
    )

    # Tipos
    df["MOB"] = df["MOB"].astype(int)
    df["pct"] = df["pct"].astype(float)

    return df


def build_demo_df_all(
    df_bgi3: pd.DataFrame,
    df_bgi4: pd.DataFrame,
    df_cast: pd.DataFrame,
) -> pd.DataFrame:
    a = _normalize_one_long(df_bgi3, tipo_mora="BGI3+")
    b = _normalize_one_long(df_bgi4, tipo_mora="BGI4+")
    c = _normalize_one_long(df_cast, tipo_mora="CAST")

    df_all = pd.concat([a, b, c], ignore_index=True)

    # Sanity check: llave única
    if df_all["folio_mora_mob_id"].duplicated().any():
        dup = df_all.loc[
            df_all["folio_mora_mob_id"].duplicated(),
            "folio_mora_mob_id"
        ].head(5).tolist()
        raise ValueError(f"Duplicados en folio_mora_mob_id. Ejemplos: {dup}")

    return df_all


In [37]:
df_dummy_all_long = build_demo_df_all(df_bgi3=df_dummy_bgi3_long, 
                                      df_bgi4=df_dummy_bgi4_long, 
                                      df_cast=df_dummy_CAST_long,)

df_dummy_all_long

,Folio,Cosecha,Capital,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,Notificaciones,Atria Plus,...,Score Buro,% Utilizacion Revolvente,Endeudamiento,MOB,pct,tipo_mora,mora_amount,mora_ever,folio_mora_id,folio_mora_mob_id
0,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,1,17.331383,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|1
1,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,2,0.541315,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|2
2,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,3,11.934600,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|3
3,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,4,10.209965,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|4
4,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,5,16.688253,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553495,4500,2025-05-01,570078,E,Ciudad Juarez,36,4 a <6,Z04,PE A,GG,...,>0 a <556,>=0 a <0.068,>=0.171 a <0.305,37,15.343634,CAST,43960.0,1,4500|CAST,4500|CAST|37
553496,4500,2025-05-01,570078,E,Ciudad Juarez,36,4 a <6,Z04,PE A,GG,...,>0 a <556,>=0 a <0.068,>=0.171 a <0.305,38,14.834832,CAST,43960.0,1,4500|CAST,4500|CAST|38
553497,4500,2025-05-01,570078,E,Ciudad Juarez,36,4 a <6,Z04,PE A,GG,...,>0 a <556,>=0 a <0.068,>=0.171 a <0.305,39,1.972026,CAST,43960.0,1,4500|CAST,4500|CAST|39
553498,4500,2025-05-01,570078,E,Ciudad Juarez,36,4 a <6,Z04,PE A,GG,...,>0 a <556,>=0 a <0.068,>=0.171 a <0.305,40,17.569931,CAST,43960.0,1,4500|CAST,4500|CAST|40


In [38]:
print(df_dummy_all_long.shape)
print(df_dummy_all_long["tipo_mora"].value_counts())

(553500, 26)
tipo_mora
BGI3+    184500
BGI4+    184500
CAST     184500
Name: count, dtype: int64


In [39]:
df_dummy_all_long.head()

,Folio,Cosecha,Capital,Perfil,Ciudad Atria,Plazo,Antiguedad Unidad,Laboratorios,Notificaciones,Atria Plus,...,Score Buro,% Utilizacion Revolvente,Endeudamiento,MOB,pct,tipo_mora,mora_amount,mora_ever,folio_mora_id,folio_mora_mob_id
0,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,1,17.331383,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|1
1,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,2,0.541315,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|2
2,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,3,11.934600,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|3
3,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,4,10.209965,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|4
4,1,2022-06-01,1112783,E,Chihuahua,12,>=8,W2,GG 3,PE,...,>663,>=0.621663,NULL,5,16.688253,BGI3+,264094.0,1,1|BGI3+,1|BGI3+|5


In [40]:
df_dummy_all_long.columns

Index(['Folio', 'Cosecha', 'Capital', 'Perfil', 'Ciudad Atria', 'Plazo',
       'Antiguedad Unidad', 'Laboratorios', 'Notificaciones', 'Atria Plus',
       'MMX 6', 'MMX 12', 'MMX 24', 'Score generico de bancos', 'Cluster Mora',
       'Cluster Experiencia', 'Score Buro', '% Utilizacion Revolvente',
       'Endeudamiento', 'MOB', 'pct', 'tipo_mora', 'mora_amount', 'mora_ever',
       'folio_mora_id', 'folio_mora_mob_id'],
      dtype='object')

In [41]:
df_dummy_all_long[['Folio', 'MOB', 'pct', 'tipo_mora', 'mora_amount', 'mora_ever']]

,Folio,MOB,pct,tipo_mora,mora_amount,mora_ever
0,1,1,17.331383,BGI3+,264094.0,1
1,1,2,0.541315,BGI3+,264094.0,1
2,1,3,11.934600,BGI3+,264094.0,1
3,1,4,10.209965,BGI3+,264094.0,1
4,1,5,16.688253,BGI3+,264094.0,1
...,...,...,...,...,...,...
553495,4500,37,15.343634,CAST,43960.0,1
553496,4500,38,14.834832,CAST,43960.0,1
553497,4500,39,1.972026,CAST,43960.0,1
553498,4500,40,17.569931,CAST,43960.0,1


In [42]:
df_dummy_all_long.to_csv("dummy_output_data/df_dummy_all_long.csv", index=False)
print('Data Frame completo en formato long guardados')

df_dummy_all_long.head(1000).to_csv("dummy_output_data/df_dummy_all_long_extract.csv", index=False)
print('Extracto de df guardado')

Data Frame completo en formato long guardados
Extracto de df guardado


In [43]:
df_dummy_all_long.Cosecha

0        2022-06-01
1        2022-06-01
2        2022-06-01
3        2022-06-01
4        2022-06-01
            ...    
553495   2025-05-01
553496   2025-05-01
553497   2025-05-01
553498   2025-05-01
553499   2025-05-01
Name: Cosecha, Length: 553500, dtype: datetime64[ns]